# Data Processing and Assortment Analysis
Cleaned portfolio version focused on:
- data preprocessing
- feature engineering
- assortment optimization
- clustering analysis
- distribution analysis

In [ ]:
import pandas as pd
import numpy as np
# Preprocessing Data before Analysis
# ---------------------------------------------------
# 1. LOAD DATA
# ---------------------------------------------------
df = pd.read_csv("All_POS_with_Attributes.csv")

print("Raw shape:", df.shape)

# ---------------------------------------------------
# 2. RENAME KEY COLUMNS TO SIMPLE NAMES
#    (only where needed – keeps it readable & consistent)
# ---------------------------------------------------
rename_map = {
    "UPC 13 digit": "UPC",
    "Aisle Name": "Aisle",
    "Category Name": "Category",
    "Sub-Category Name": "Sub_Category",
    "Manufacturer Name": "Manufacturer",
    "Brand Franchise Name": "Brand_Franchise",
    "Brand Name": "Brand",
    "Flavor / Scent": "Flavor_Scent",
    "Type Of Meat Substituted": "Type_Of_Meat_Substituted",
    "Type Of Substitute": "Type_Of_Substitute",
    "ACV Weighted Distribution": "ACV_Weighted_Distribution"
}

df = df.rename(columns=rename_map)

# ---------------------------------------------------
# 3. PARSE TIME COLUMN → WEEK (datetime)
#    You already changed Time to look like "01-12-20"
# ---------------------------------------------------
df["Week"] = pd.to_datetime(df["Time"], format="%m-%d-%y")

# (Optional) keep Time or drop it; I like to keep both
# df = df.drop(columns=["Time"])

# ---------------------------------------------------
# 4. BASIC COLUMN NAME CLEANUP (only cosmetic)
#    - lower case
#    - replace spaces with underscores
#    - remove % and slashes
# ---------------------------------------------------
df.columns = (
    df.columns
      .str.strip()
      .str.replace(" ", "_")
      .str.replace("/", "_", regex=False)
      .str.replace("%", "pct", regex=False)
      .str.lower()
)

print("Cleaned columns:")
print(df.columns.tolist())

# After this, examples:
# "Unit Sales" -> "unit_sales"
# "Dollar Sales" -> "dollar_sales"
# "ACV Weighted Distribution" -> "acv_weighted_distribution"
# "Total Ounces" -> "total_ounces"
# "Type Of Substitute" -> "type_of_substitute"

# ---------------------------------------------------
# 5. ENSURE NUMERIC TYPES FOR MEASURES
#    (Everything Circana calls sales/price/ACV)
# ---------------------------------------------------
numeric_cols = [
    # core measures
    "unit_sales", "volume_sales", "dollar_sales",
    "price_per_unit", "price_per_volume",
    "acv_weighted_distribution",
    "base_unit_sales", "base_volume_sales", "base_dollar_sales",
    "incremental_units", "incremental_volume", "incremental_dollars",
    # merch splits (only converted if present)
    "unit_sales_no_merch", "unit_sales_any_merch",
    "unit_sales_price_reductions_only", "unit_sales_feature_only",
    "unit_sales_display_only", "unit_sales_special_pack_only",
    "unit_sales_feature_and_display",
    "volume_sales_no_merch", "volume_sales_any_merch",
    "volume_sales_price_reductions_only", "volume_sales_feature_only",
    "volume_sales_display_only", "volume_sales_special_pack_only",
    "volume_sales_feature_and_display",
    "dollar_sales_no_merch", "dollar_sales_any_merch",
    "dollar_sales_price_reductions_only", "dollar_sales_feature_only",
    "dollar_sales_display_only", "dollar_sales_special_pack_only",
    "dollar_sales_feature_and_display",
    "acv_weighted_distribution_no_merch",
    "acv_weighted_distribution_any_merch",
    "acv_weighted_distribution_price_reductions_only",
    "acv_weighted_distribution_feature_only",
    "acv_weighted_distribution_display_only",
    "acv_weighted_distribution_special_pack_only",
    "acv_weighted_distribution_feature_and_display",
]

for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# For sales & volumes, NaN basically means "no sales" → set to 0
sales_like = [c for c in numeric_cols if "sales" in c or "units" in c or "volume" in c]
for col in sales_like:
    if col in df.columns:
        df[col] = df[col].fillna(0)

# Leave prices & ACV as NaN if missing (better than forcing 0)
# ---------------------------------------------------
# 6. BASIC ATTRIBUTE CLEANUP (strings)
# ---------------------------------------------------
cat_cols = [
    "upc", "product", "aisle", "category", "sub_category",
    "manufacturer", "brand_franchise", "brand",
    "package", "form", "flavor_scent", "meat_source",
    "product_type", "type_of_meat_substituted",
    "type_of_substitute", "cooked_info"
]

for col in cat_cols:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip()

# ---------------------------------------------------
# 7. SIMPLE TIME FEATURES FOR MODELING
# ---------------------------------------------------
df["year"] = df["week"].dt.year
df["month"] = df["week"].dt.month
df["quarter"] = df["week"].dt.quarter
df["week_number"] = df["week"].dt.isocalendar().week.astype(int)

# ---------------------------------------------------
# 8. OPTIONAL: LOG TRANSFORMS (useful for regressions)
# ---------------------------------------------------
for col in ["unit_sales", "volume_sales", "dollar_sales",
            "price_per_unit", "price_per_volume",
            "acv_weighted_distribution"]:
    if col in df.columns:
        df[f"log_{col}"] = np.log(df[col] + 1)

print("Final shape after light preprocessing:", df.shape)

#### Q1 – Attribute interactions: Which combinations of attributes (form, flavor, type, pack, size, etc.) drive sales?
- Model Used: Random Forest + feature importance (+ optional SHAP).

In [ ]:
# ============================================
# Q1: ATTRIBUTE INTERACTION MODEL
# ============================================
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error
import matplotlib.pyplot as plt

# 1) Aggregate to UPC-level (across all weeks)
agg_cols = {
    "unit_sales": "sum",
    "volume_sales": "sum",
    "dollar_sales": "sum",
    "price_per_unit": "mean",
    "price_per_volume": "mean",
    "acv_weighted_distribution": "mean",
    "total_ounces": "first",
    "total_count": "first",
    "form": "first",
    "flavor_scent": "first",
    "product_type": "first",
    "type_of_substitute": "first",
    "cooked_info": "first",
    "package": "first",
    "brand": "first",
}

df_upc = (
    df.groupby("upc", as_index=False)
      .agg(agg_cols)
)

# Target variable: total unit sales per UPC (log helps handle skew)
df_upc = df_upc[df_upc["unit_sales"] > 0].copy()
df_upc["log_unit_sales"] = np.log(df_upc["unit_sales"])

# 2) Select features
feature_cols_cat = [
    "form", "flavor_scent", "product_type",
    "type_of_substitute", "cooked_info",
    "package", "brand"
]

feature_cols_num = [
    "total_ounces", "total_count",
    "price_per_unit", "price_per_volume",
    "acv_weighted_distribution"
]

X = df_upc[feature_cols_cat + feature_cols_num]
y = df_upc["log_unit_sales"]

# 3) Build preprocessing + model pipeline
cat_transformer = OneHotEncoder(handle_unknown="ignore")
preprocess = ColumnTransformer(
    transformers=[
        ("cat", cat_transformer, feature_cols_cat),
        ("num", "passthrough", feature_cols_num),
    ]
)

rf_model = RandomForestRegressor(
    n_estimators=400,
    max_depth=12,
    min_samples_leaf=10,
    random_state=42,
    n_jobs=-1,
)

pipe = Pipeline(steps=[
    ("prep", preprocess),
    ("model", rf_model)
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

pipe.fit(X_train, y_train)
y_pred = pipe.predict(X_test)

print("R² on test:", r2_score(y_test, y_pred))
print("RMSE on test:", np.sqrt(mean_squared_error(y_test, y_pred)))

# 4) Feature importance (top 25 one-hot features)
# Get names of one-hot encoded cat features
ohe_feature_names = pipe.named_steps["prep"] \
                         .named_transformers_["cat"] \
                         .get_feature_names_out(feature_cols_cat)

all_feature_names = list(ohe_feature_names) + feature_cols_num

importances = pipe.named_steps["model"].feature_importances_
fi = (
    pd.DataFrame({"feature": all_feature_names, "importance": importances})
      .sort_values("importance", ascending=False)
      .head(25)
)

print(fi)

plt.figure(figsize=(8, 10))
plt.barh(fi["feature"][::-1], fi["importance"][::-1])
plt.xlabel("Importance")
plt.title("Top 25 Feature Importances – Attribute + Econ Model")
plt.tight_layout()
plt.show()

# OPTIONAL: SHAP (comment out if you don't need it)
try:
    import shap

    # Sample some rows for speed
    X_sample = X_test.sample(min(200, len(X_test)), random_state=42)

    # Transform with the same preprocessing as the model uses
    X_sample_t = pipe.named_steps["prep"].transform(X_sample)

    # If it's a sparse matrix, convert to dense
    if hasattr(X_sample_t, "toarray"):
        X_sample_t = X_sample_t.toarray()

    # Ensure numeric dtype
    X_sample_t = X_sample_t.astype(float)

    # Build explainer and compute SHAP values
    explainer = shap.TreeExplainer(pipe.named_steps["model"])
    shap_values = explainer.shap_values(X_sample_t)

    shap.summary_plot(
        shap_values,
        X_sample_t,
        feature_names=all_feature_names
    )

except ImportError:
    print("shap is not installed; skipping SHAP plots.")
except Exception as e:
    print("Could not compute SHAP values, skipping SHAP plots.")
    print("Reason:", e)

#### Q3 – Assortment / SKU efficiency: Which SKUs are core, niche, expansion opportunities, or delist candidates?
- Aggregate to UPC, build velocity & distribution metrics, then cluster.

In [ ]:
# ============================================
# Q3A: UPC-LEVEL PERFORMANCE TABLE
# ============================================
import pandas as pd
import numpy as np

# Number of weeks in dataset (for normalization)
total_weeks = df["week"].nunique()

upc_perf = (
    df.groupby(["upc", "brand", "form", "package", "total_ounces", "total_count"],
               as_index=False)
      .agg({
          "unit_sales": "sum",
          "volume_sales": "sum",
          "dollar_sales": "sum",
          "acv_weighted_distribution": "mean",
          "price_per_unit": "mean"
      })
)

# Basic derived metrics
upc_perf["unit_sales_per_week"] = upc_perf["unit_sales"] / total_weeks
upc_perf["volume_sales_per_week"] = upc_perf["volume_sales"] / total_weeks

# Velocity: units per week per point of ACV (approximation)
upc_perf["unit_velocity_per_acv"] = (
    upc_perf["unit_sales_per_week"] /
    upc_perf["acv_weighted_distribution"].replace(0, np.nan)
)

# Handle inf/nan
upc_perf["unit_velocity_per_acv"] = upc_perf["unit_velocity_per_acv"].replace([np.inf, -np.inf], np.nan)
upc_perf["unit_velocity_per_acv"] = upc_perf["unit_velocity_per_acv"].fillna(0)

print(upc_perf.head())

In [ ]:
# ============================================
# Q3B: K-MEANS CLUSTERING FOR ASSORTMENT
# ============================================
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt

# Choose numeric features for clustering
cluster_features = upc_perf[[
    "unit_velocity_per_acv",
    "acv_weighted_distribution",
    "price_per_unit",
    "total_ounces"
]].copy()

# Replace any remaining NaN
cluster_features = cluster_features.fillna(0)

# Scale for k-means
scaler = StandardScaler()
X_scaled = scaler.fit_transform(cluster_features)

# Try k=4 clusters (Core, Expansion, Niche, Underperformer)
kmeans = KMeans(n_clusters=4, random_state=42, n_init=20)
clusters = kmeans.fit_predict(X_scaled)

upc_perf["assortment_cluster"] = clusters

# Inspect cluster profiles
cluster_profile = (
    upc_perf.groupby("assortment_cluster")
            .agg({
                "unit_velocity_per_acv": "mean",
                "acv_weighted_distribution": "mean",
                "price_per_unit": "mean",
                "total_ounces": "mean",
                "unit_sales_per_week": "mean",
            })
            .reset_index()
)

print(cluster_profile)

# Quick 2D scatter: velocity vs ACV, colored by cluster
plt.figure(figsize=(6,5))
for c in sorted(upc_perf["assortment_cluster"].unique()):
    sub = upc_perf[upc_perf["assortment_cluster"] == c]
    plt.scatter(
        sub["acv_weighted_distribution"],
        sub["unit_velocity_per_acv"],
        s=20,
        alpha=0.6,
        label=f"Cluster {c}"
    )

plt.xlabel("ACV Weighted Distribution (mean)")
plt.ylabel("Unit Velocity per ACV")
plt.title("UPC Clusters – Distribution vs Velocity")
plt.legend()
plt.ylim(0, 5000)   # adjust as needed
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================
# Q3C: HUMAN-READABLE CLUSTER LABELS
# ============================================
# Example: assign labels based on velocity & ACV averages
# (You can adjust thresholds based on cluster_profile results.)

# Start by ranking clusters by velocity
cluster_profile = cluster_profile.sort_values("unit_velocity_per_acv", ascending=False)
cluster_profile["velocity_rank"] = range(1, len(cluster_profile) + 1)

# Map cluster → label manually after inspecting cluster_profile
cluster_label_map = {
    # UPDATE THESE after you inspect cluster_profile
    0: "TBD",
    1: "TBD",
    2: "TBD",
    3: "TBD",
}

upc_perf["assortment_label"] = upc_perf["assortment_cluster"].map(cluster_label_map)

# Example: filter to Gardein SKUs and see their roles
gardein_upcs = upc_perf[upc_perf["brand"].str.contains("GARDEIN", case=False)]
print(gardein_upcs[[
    "upc", "brand", "form", "package",
    "unit_velocity_per_acv",
    "acv_weighted_distribution",
    "price_per_unit",
    "assortment_cluster",
    "assortment_label"
]].head(20))

In [ ]:
# Q1-A. Total Unit Sales by Form
form_perf = (
    df.groupby("form", as_index=False)["unit_sales"]
      .sum()
      .sort_values("unit_sales", ascending=False)
)

plt.figure(figsize=(8,4))
plt.bar(form_perf["form"], form_perf["unit_sales"])
plt.xticks(rotation=45, ha="right")
plt.title("Total Unit Sales by Form")
plt.xlabel("Form")
plt.ylabel("Unit Sales")
plt.tight_layout()
plt.show()

In [ ]:
# Q1-B. Top 15 Flavors by Sales
flavor_perf = (
    df.groupby("flavor_scent", as_index=False)["unit_sales"]
      .sum()
      .sort_values("unit_sales", ascending=False)
      .head(15)
)

plt.figure(figsize=(8,4))
plt.bar(flavor_perf["flavor_scent"], flavor_perf["unit_sales"])
plt.xticks(rotation=45, ha="right")
plt.title("Top 15 Flavors by Unit Sales")
plt.xlabel("Flavor")
plt.ylabel("Unit Sales")
plt.tight_layout()
plt.show()

In [ ]:
# Q1-C. Total Unit Sales by Package Type
package_perf = (
    df.groupby("package", as_index=False)["unit_sales"]
      .sum()
      .sort_values("unit_sales", ascending=False)
)

plt.figure(figsize=(8,4))
plt.bar(package_perf["package"], package_perf["unit_sales"])
plt.xticks(rotation=45, ha="right")
plt.title("Total Unit Sales by Package Type")
plt.xlabel("Package Type")
plt.ylabel("Unit Sales")
plt.tight_layout()
plt.show()

In [ ]:
# Q1-D. Total Unit Sales by Type of Substitute
subtype_perf = (
    df.groupby("type_of_substitute", as_index=False)["unit_sales"]
      .sum()
      .sort_values("unit_sales", ascending=False)
)

plt.figure(figsize=(8,4))
plt.bar(subtype_perf["type_of_substitute"], subtype_perf["unit_sales"])
plt.xticks(rotation=45, ha="right")
plt.title("Total Unit Sales by Type of Substitute")
plt.xlabel("Type of Substitute")
plt.ylabel("Unit Sales")
plt.tight_layout()
plt.show()

In [ ]:
# Q1-E. ACV vs Unit Sales Scatterplot (Category)
plt.figure(figsize=(6,5))
plt.scatter(
    df.groupby("upc")["acv_weighted_distribution"].mean(),
    df.groupby("upc")["unit_sales"].sum(),
    s=20, alpha=0.6
)
plt.xlabel("ACV Weighted Distribution (mean)")
plt.ylabel("Total Unit Sales")
plt.title("ACV vs Total Unit Sales (per UPC)")
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# Q1-F. Price vs Unit Sales Scatterplot
upc_summary = (
    df.groupby("upc", as_index=False)
      .agg({"price_per_unit":"mean", "unit_sales":"sum"})
)

plt.figure(figsize=(6,5))
plt.scatter(upc_summary["price_per_unit"], upc_summary["unit_sales"], alpha=0.6)
plt.xlabel("Price per Unit")
plt.ylabel("Total Unit Sales")
plt.title("Price vs Unit Sales (per UPC)")
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# Q3-A. UPC Clusters – ACV vs Velocity Scatter (You already have this)
plt.figure(figsize=(7,5))
for c in sorted(upc_perf["assortment_cluster"].unique()):
    sub = upc_perf[upc_perf["assortment_cluster"] == c]
    plt.scatter(
        sub["acv_weighted_distribution"],
        sub["unit_velocity_per_acv"],
        s=20, alpha=0.7,
        label=f"Cluster {c}"
    )

plt.xlabel("ACV Weighted Distribution (mean)")
plt.ylabel("Unit Velocity per ACV")
plt.title("UPC Clusters – Distribution vs Velocity")
plt.legend()
plt.ylim(0, 5000)   # adjust as needed
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# Q3-B. SKU Count per Cluster
cluster_counts = upc_perf["assortment_cluster"].value_counts().sort_index()

plt.figure(figsize=(6,4))
plt.bar(cluster_counts.index.astype(str), cluster_counts.values)
plt.xlabel("Cluster")
plt.ylabel("Number of SKUs")
plt.title("Number of SKUs per Assortment Cluster")
plt.tight_layout()
plt.show()

In [ ]:
# Q3-C. Gardein SKUs per Cluster
gardein_mask = upc_perf["brand"].str.upper().str.contains("GARDEIN")
gardein_clusters = upc_perf[gardein_mask]["assortment_cluster"].value_counts().sort_index()

plt.figure(figsize=(6,4))
plt.bar(gardein_clusters.index.astype(str), gardein_clusters.values)
plt.xlabel("Cluster")
plt.ylabel("Count of Gardein SKUs")
plt.title("Gardein Assortment Distribution by Cluster")
plt.tight_layout()
plt.show()

In [ ]:
# Q3-D. Cluster Distribution: Gardein vs Other Brands
upc_perf["is_gardein"] = upc_perf["brand"].str.upper().str.contains("GARDEIN")

cluster_gardein = (
    upc_perf.groupby(["assortment_cluster", "is_gardein"])["upc"]
            .nunique()
            .unstack(fill_value=0)
            .rename(columns={False:"Other Brands", True:"Gardein"})
)

print(cluster_gardein)

In [ ]:
# Q3-E. ACV Trend Over Time – Gardein vs Impossible
focus_brands = ["GARDEIN", "IMPOSSIBLE"]

df_acv = df[df["brand"].str.upper().isin(focus_brands)].copy()
df_acv["brand_clean"] = df_acv["brand"].str.upper()

acv_trend = (
    df_acv.groupby(["week", "brand_clean"], as_index=False)["acv_weighted_distribution"]
          .mean()
          .sort_values(["week", "brand_clean"])
)

plt.figure(figsize=(10,4))
for b in focus_brands:
    sub = acv_trend[acv_trend["brand_clean"] == b]
    plt.plot(sub["week"], sub["acv_weighted_distribution"], label=b)

plt.title("ACV Weighted Distribution Over Time – Gardein vs Impossible")
plt.xlabel("Week")
plt.ylabel("ACV Weighted Distribution (%)")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()